# Bologna and the plain: a city that sinks

Pumping groundwater out of the plain makes the ground below Bologna **subside**: in the
past by several centimetres a year. How much is left of it in 2020-2025?

**Downloads:** ~270 MB (3 L3 tiles, vertical component), nothing afterwards.

*Data: EGMS, © European Union, Copernicus Land Monitoring Service. Anything you publish from it must credit the source.*

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer

from egms_client import EgmsArchive, Query

# The key comes from EGMS_SERVICE_KEY or EGMS_SERVICE_KEY_FILE (see the README).
# Downloads are kept here: run the notebook again and nothing is downloaded.
DATA = Path(os.environ.get("EGMS_DATA_DIR", "egms-data"))
RELEASE = "2020-2025"
to_laea = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)

In [ ]:
from egms_client.mosaic import lonlat_to_laea, mosaic, to_xarray

BOX = (11.0, 44.35, 11.8, 44.75)
query = Query(levels=("L3",), release=RELEASE, bbox=BOX, product_type="ORTHO-UP")

async with EgmsArchive() as archive:
    tiffs = await archive.fetch(query, DATA, keep=["tiff"])

minx, miny, maxx, maxy = lonlat_to_laea(*BOX)
grid = to_xarray(mosaic(tiffs, minx=minx, miny=miny, maxx=maxx, maxy=maxy))
print(f"{int(grid.notnull().sum()):,} cells of 100 m with a measurement")

In [ ]:
BOLOGNA = to_laea.transform(11.343, 44.494)

fig, ax = plt.subplots(figsize=(11, 6))
grid.plot(
    ax=ax, cmap="RdBu_r", vmin=-25, vmax=25, cbar_kwargs={"label": "vertical speed (mm/year)"}
)
ax.plot(*BOLOGNA, "k+", markersize=14)
ax.annotate("Bologna", BOLOGNA, xytext=(8, 8), textcoords="offset points")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Vertical ground speed, Bologna and the plain, 2020-2025 (blue: sinking)")

## How much of the area sinks, and how fast

In [ ]:
values = grid.values[np.isfinite(grid.values)]
for limit in (2, 5, 10):
    print(
        f"sinking faster than {limit:>2} mm/year: {np.mean(values < -limit):6.1%} of measured cells"
    )

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(values, bins=np.arange(-40, 15, 1), color="#2a78d6")
ax.axvline(0, color="#52514e", lw=1)
ax.set_xlabel("vertical speed (mm/year)")
ax.set_ylabel("cells of 100 m")
ax.set_title("Most of the plain moves a little, a tail sinks fast")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.4, 1]})
grid.plot(ax=left, cmap="RdBu_r", vmin=-25, vmax=25, cbar_kwargs={"label": "mm/year"})
left.plot(*BOLOGNA, "k+", markersize=12)
left.set_aspect("equal")
left.set_axis_off()
left.set_title("Vertical speed, Bologna and the plain")
right.hist(values, bins=np.arange(-40, 15, 1), color="#2a78d6")
right.axvline(0, color="#52514e", lw=1)
right.set_xlabel("mm/year")
right.set_title("How fast the cells move")
fig.text(
    0.01,
    0.01,
    "EGMS Ortho L3 2020-2025. Contains modified Copernicus Land Monitoring Service information.",
    fontsize=8,
)
fig.savefig("../docs/images/notebook-sinking-bologna.png", dpi=100, bbox_inches="tight")